In [ ]:
import os, math
from pathlib import Path
import scanpy as sc
import numpy as np
import seaborn as sns
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from tqdm import tqdm
from scipy import stats
from scipy.stats import pearsonr, spearmanr, gaussian_kde
from scipy.sparse import issparse

pd.set_option("display.max_rows", 500)

In [2]:
%load_ext autoreload
%autoreload 1

%aimport s7_helpers
import s7_helpers

In [3]:
obs = pd.read_csv("./6.3.6_obs_sp_1st_2nd_scores.csv", index_col=0)
obs.sample(2)

,GSE_id,GSM_id,Age,Tissue_global,cell_type_final,sp_score_sigs,Age_bin,sp_score_sigs_dw,n_counts,n_expressed_genes
CCGTTCATCATATCGG-1-GSM7103319,GSE227136,GSM7103319,60.0,Lung,Capillary endothelial cell,-0.009274,51,-0.003344,753.0,562
AGCGCTGAGAATCCCT-1-GSM6614349,GSE214695,GSM6614349,63.0,Colon,Plasma cell,-0.064257,61,NaN,38790.0,3402


In [5]:
raw_score_col = "sp_score_sigs_dw"
out_dir = "./corr_outputs/senepy_2nd_dw_re"
os.makedirs(out_dir, exist_ok=True)

### General

In [6]:
obs_filt = obs.dropna(subset=raw_score_col)

In [7]:
obs.shape, obs_filt.shape

((1454732, 10), (921488, 10))

In [ ]:
obs_filt = obs_filt[obs_filt.n_counts < 75000]

# где >= 100 клеток
obs_filt = obs_filt.groupby(["Tissue_global", "cell_type_final", "GSM_id"]).filter(
    lambda x: len(x) >= 100
)

# где >= 3 уникальных GSM_id
obs_filt = obs_filt.groupby(["Tissue_global", "cell_type_final"]).filter(
    lambda x: x["GSM_id"].nunique() >= 3
)


for col in ["Tissue_global", "cell_type_final"]:
    obs_filt[col] = obs_filt[col].astype("category").cat.remove_unused_categories()

print(f"Отфильтровано {len(obs) - len(obs_filt)} клеток из {len(obs)}")

Отфильтровано 577792 клеток из 1454732


In [8]:
# посмотреть сырые скоры - нужно ли нормировать в один диапазон?
s7_helpers.plot_violins_by_gsm(
    obs_filt.sort_values(by=["Tissue_global", "cell_type_final"]),
    score_col=raw_score_col,
    grid=True,
)

In [ ]:
obs_filt["centered_median"] = obs_filt[raw_score_col] - obs_filt.groupby(
    ["Tissue_global", "cell_type_final", "GSM_id"]
)[raw_score_col].transform("median")

### raw score Means corr

In [14]:
stats_dict = s7_helpers.count_plot_sen_corrs(
    obs_filt, raw_score_col, count_means_corr=True
)

df_stats = pd.DataFrame.from_dict(stats_dict, orient="index")
df_stats.index = pd.MultiIndex.from_tuples(
    df_stats.index, names=["Tissue_global", "cell_type_final"]
)
df_stats = df_stats.reset_index()
df_stats.to_csv(
    os.path.join(out_dir, f"raw_{raw_score_col}_means_corr.csv"), index=False
)

### Median centering

#### GMM, k=2

In [ ]:
# thr per ct
groups = obs_filt.groupby(["Tissue_global", "cell_type_final"], sort=True)
obs_filt["is_sen_GMMk2_median_ct"] = False
for (t, ct), df in groups:
    print(f"\n\n{t}-{ct}")
    thr = s7_helpers.binarization_wrap(df, score_col="centered_median")  # thr is float
    if thr is None:
        obs_filt.loc[df.index, "is_sen_GMMk2_median_ct"] = pd.NA
    else:
        obs_filt.loc[df.index, "is_sen_GMMk2_median_ct"] = df["centered_median"] > thr

In [ ]:
# thr per gse
groups = obs_filt.groupby(["Tissue_global", "cell_type_final", "GSE_id"], sort=True)
obs_filt["is_sen_GMMk2_median_gse"] = False
for (t, ct, gse), df in groups:
    print(f"\n\n{t}-{ct}-{gse}")
    thr = s7_helpers.binarization_wrap(df, score_col="centered_median")  # thr is float
    if thr is None:
        obs_filt.loc[df.index, "is_sen_GMMk2_median_gse"] = pd.NA
    else:
        obs_filt.loc[df.index, "is_sen_GMMk2_median_gse"] = df["centered_median"] > thr

In [ ]:
# thr per donor
groups = obs_filt.groupby(
    ["Tissue_global", "cell_type_final", "GSE_id", "GSM_id"], sort=True
)
obs_filt["is_sen_GMMk2_median_gsm"] = False
for (t, ct, gse, gsm), df in groups:
    print(f"\n\n{t}-{ct}-{gse}-{gsm}-{int(df.Age.iloc[0])} y.o.")
    thr = s7_helpers.binarization_wrap(df, score_col="centered_median")  # thr is float
    if thr is None:
        obs_filt.loc[df.index, "is_sen_GMMk2_median_gsm"] = pd.NA
    else:
        obs_filt.loc[df.index, "is_sen_GMMk2_median_gsm"] = df["centered_median"] > thr

In [ ]:
# # ----------рисуем все графики сразу и сохраняем статистику в csv --------------------
for score_col in [
    "is_sen_GMMk2_median_ct",
    # "is_sen_GMMk2_median_gse",
    # "is_sen_GMMk2_median_gsm",
]:
    print("=" * 100, "\n", " " * 40, score_col, "\n", "=" * 100)
    stats_dict = s7_helpers.count_plot_sen_corrs(obs_filt, score_col)

    df_stats = pd.DataFrame.from_dict(stats_dict, orient="index")
    df_stats.index = pd.MultiIndex.from_tuples(
        df_stats.index, names=["Tissue_global", "cell_type_final"]
    )
    df_stats = df_stats.reset_index()
    df_stats.to_csv(os.path.join(out_dir, f"{score_col}.csv"), index=False)

#### GMM, k=3

In [ ]:
# thr per ct
groups = obs_filt.groupby(["Tissue_global", "cell_type_final"], sort=True)
obs_filt["is_sen_GMMk3_median_ct"] = False
for (t, ct), df in groups:
    print(f"\n\n{t}-{ct}")
    thr = s7_helpers.binarize_df_GMMk3(df, score_col="centered_median")  # thr is float
    if thr is None:
        obs_filt.loc[df.index, "is_sen_GMMk3_median_ct"] = pd.NA
    else:
        obs_filt.loc[df.index, "is_sen_GMMk3_median_ct"] = df["centered_median"] > thr

In [ ]:
# thr per gse
groups = obs_filt.groupby(["Tissue_global", "cell_type_final", "GSE_id"], sort=True)
obs_filt["is_sen_GMMk3_median_gse"] = False
for (t, ct, gse), df in groups:
    print(f"\n\n{t}-{ct}-{gse}")
    thr = s7_helpers.binarize_df_GMMk3(df, score_col="centered_median")  # thr is float
    if thr is None:
        obs_filt.loc[df.index, "is_sen_GMMk3_median_gse"] = pd.NA
    else:
        obs_filt.loc[df.index, "is_sen_GMMk3_median_gse"] = df["centered_median"] > thr

In [ ]:
# thr per donor
groups = obs_filt.groupby(
    ["Tissue_global", "cell_type_final", "GSE_id", "GSM_id"], sort=True
)
obs_filt["is_sen_GMMk3_median_gsm"] = False
for (t, ct, gse, gsm), df in groups:
    print(f"\n\n{t}-{ct}-{gse}-{gsm}-{int(df.Age.iloc[0])} y.o.")
    thr = s7_helpers.binarize_df_GMMk3(df, score_col="centered_median")  # thr is float
    if thr is None:
        obs_filt.loc[df.index, "is_sen_GMMk3_median_gsm"] = pd.NA
    else:
        obs_filt.loc[df.index, "is_sen_GMMk3_median_gsm"] = df["centered_median"] > thr

In [ ]:
# # ----------рисуем все графики сразу и сохраняем статистику в csv --------------------
for score_col in [
    "is_sen_GMMk3_median_ct",
    "is_sen_GMMk3_median_gse",
    "is_sen_GMMk3_median_gsm",
]:
    print("=" * 100, "\n", " " * 40, score_col, "\n", "=" * 100)
    stats_dict = s7_helpers.count_plot_sen_corrs(obs_filt, score_col)

    df_stats = pd.DataFrame.from_dict(stats_dict, orient="index")
    df_stats.index = pd.MultiIndex.from_tuples(
        df_stats.index, names=["Tissue_global", "cell_type_final"]
    )
    df_stats = df_stats.reset_index()
    df_stats.to_csv(os.path.join(out_dir, f"{score_col}.csv"), index=False)

In [30]:
obs_filt.drop(
    columns=[
        "GSE_id",
        "GSM_id",
        "Age",
        "Tissue_global",
        "cell_type_final",
        "sp_score_sigs",
        "Age_bin",
        "n_counts",
        "n_expressed_genes",
    ],
    inplace=True,
)

In [31]:
obs_filt.to_csv("./sp_2nd_scores_is_sen_labels.csv", index=True)